# KS0223: mini-обучение и езда через ROS2

Ноутбук показывает цикл `данные -> простая модель -> управление` через ROS2 topic `/cmd_vel`.
Требуется запущенный demo-контур (`make demo-control`) и Unity в `Play`.


In [1]:
%pip install -q requests numpy matplotlib pandas


Note: you may need to restart the kernel to use updated packages.


In [2]:
from __future__ import annotations

import ast
import os
import re
import subprocess
import time
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import requests


def find_repo_root(start: Path) -> Path:
    cur = start.resolve()
    for candidate in [cur, *cur.parents]:
        if (candidate / "README.md").exists() and (candidate / "Makefile").exists():
            return candidate
    raise RuntimeError("Не найден корень репозитория.")


ROOT = find_repo_root(Path.cwd())
BASE_URL = os.getenv("UAVSIM_BASE_URL", "http://127.0.0.1:8000")
ROS_CONTAINER = os.getenv("UAVSIM_ROS_CONTAINER", "uavsim-ros2-desktop")
NS = os.getenv("UAVSIM_ROS_NAMESPACE", "/uavsim/ks0223")

print("ROOT      :", ROOT)
print("API       :", BASE_URL)
print("ROS ns    :", NS)
print("container :", ROS_CONTAINER)


ROOT      : <repo>
API       : http://127.0.0.1:8000
ROS ns    : /uavsim/ks0223
container : uavsim-ros2-desktop


In [3]:
def run_cmd(cmd: list[str], cwd: Path | None = None, timeout: int = 20) -> str:
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd else None,
        capture_output=True,
        text=True,
        timeout=timeout,
    )
    if proc.returncode != 0:
        raise RuntimeError(
            f"Command failed ({proc.returncode}): {' '.join(cmd)}\nSTDERR:\n{proc.stderr}"
        )
    return proc.stdout


def ros_exec(inner_cmd: str, timeout: int = 20) -> str:
    wrapped = f'su - ubuntu -c "source /opt/ros/humble/setup.bash; {inner_cmd}"'
    cmd = ["docker", "exec", ROS_CONTAINER, "bash", "-lc", wrapped]
    return run_cmd(cmd, timeout=timeout)


def ros_read_array(topic: str) -> list[float]:
    out = ros_exec(f"timeout 4 ros2 topic echo {topic} --once --field data", timeout=8)
    m = re.search(r"\[([^\]]+)\]", out)
    if not m:
        raise RuntimeError(f"Не удалось распарсить массив из: {out}")
    values = ast.literal_eval("[" + m.group(1) + "]")
    return [float(v) for v in values]


def ros_read_scalar(topic: str) -> float:
    out = ros_exec(f"timeout 4 ros2 topic echo {topic} --once --field data", timeout=8)
    out = out.replace("---", "").strip()
    return float(out.splitlines()[0].strip())


def ros_cmd_vel(linear: float, angular: float) -> None:
    cmd = [
        "make",
        "ros-cmd-vel",
        f"UAVSIM_CMD_LINEAR={linear}",
        f"UAVSIM_CMD_ANGULAR={angular}",
    ]
    run_cmd(cmd, cwd=ROOT, timeout=20)


def ros_stop() -> None:
    run_cmd(["make", "ros-stop"], cwd=ROOT, timeout=20)


In [4]:
# Проверка окружения
RUNTIME_READY = False
try:
    health = requests.get(f"{BASE_URL}/health", timeout=5).json()
    print("health:", health)

    topics = ros_exec(f"ros2 topic list | grep '^{NS}' | sort", timeout=10)
    print(topics)
    RUNTIME_READY = True
except Exception as exc:
    print("Контур пока недоступен:", exc)
    print("Нужно: Unity в Play + make demo-control")


Контур пока недоступен: HTTPConnectionPool(host='127.0.0.1', port=8000): Max retries exceeded with url: /health (Caused by NewConnectionError("HTTPConnection(host='127.0.0.1', port=8000): Failed to establish a new connection: [Errno 61] Connection refused"))
Нужно: Unity в Play + make demo-control


In [5]:
if not RUNTIME_READY:
    print("Пропуск reset: runtime недоступен.")
else:
    # Сброс симуляции в baseline
    reset_cfg = {
        "seed": 1,
        "timeScale": 1.0,
        "selectedTrackId": "track.basic_arena.v1",
        "selectedVehicleId": "vehicle.ks0223.v1",
        "trackParams": [],
        "vehicleParams": [],
        "flags": [],
    }
    _ = requests.post(f"{BASE_URL}/reset", json=reset_cfg, timeout=10).json()
    time.sleep(0.4)
    print("reset done")


Пропуск reset: runtime недоступен.


In [6]:
# Сбор мини-датасета с line_tracker через ROS2
X = np.empty((0, 8), dtype=float)
y = np.empty((0,), dtype=float)
samples = []

if not RUNTIME_READY:
    print("Пропуск сбора данных: runtime недоступен.")
else:
    rng = np.random.default_rng(42)
    rows = []
    targets = []

    for _ in range(28):
        ang_probe = float(rng.uniform(-0.45, 0.45))
        ros_cmd_vel(linear=0.25, angular=ang_probe)
        time.sleep(0.18)

        line = ros_read_array(f"{NS}/line_tracker/front_norm")
        speed = ros_read_scalar(f"{NS}/speedometer/mps")

        # Простое teacher-правило: баланс левых/правых датчиков
        teacher = float(np.clip((line[0] + line[1]) - (line[3] + line[4]), -1.0, 1.0) * 0.6)

        feat = [1.0, line[0], line[1], line[2], line[3], line[4], line[0] - line[4], line[1] - line[3]]
        rows.append(feat)
        targets.append(teacher)
        samples.append({"line": line, "speed": speed, "teacher_ang": teacher})

    ros_stop()
    X = np.asarray(rows, dtype=float)
    y = np.asarray(targets, dtype=float)

print("dataset:", X.shape, "target:", y.shape)
if samples:
    print("example:", samples[0])


Пропуск сбора данных: runtime недоступен.
dataset: (0, 8) target: (0,)


In [7]:
# Обучаем линейную модель
if len(X) == 0:
    w = np.zeros(8, dtype=float)
    mse = float("nan")
    print("Пропуск обучения: нет данных.")
else:
    w, *_ = np.linalg.lstsq(X, y, rcond=None)
    pred = X @ w
    mse = float(np.mean((pred - y) ** 2))
    print("weights:", np.round(w, 4))
    print("train MSE:", round(mse, 6))


Пропуск обучения: нет данных.


In [8]:
# Rollout обученной политики через ROS2 /cmd_vel
history = []

if not RUNTIME_READY:
    print("Пропуск rollout: runtime недоступен.")
else:
    for t in range(42):
        line = ros_read_array(f"{NS}/line_tracker/front_norm")
        feat = np.array(
            [1.0, line[0], line[1], line[2], line[3], line[4], line[0] - line[4], line[1] - line[3]],
            dtype=float,
        )
        angular = float(np.clip(feat @ w, -0.8, 0.8))
        linear = float(np.clip(0.20 + 0.18 * line[2], 0.18, 0.38))

        ros_cmd_vel(linear=linear, angular=angular)
        time.sleep(0.16)
        speed = ros_read_scalar(f"{NS}/speedometer/mps")

        history.append(
            {
                "step": t,
                "linear": linear,
                "angular": angular,
                "speed": speed,
                "center": line[2],
            }
        )

    ros_stop()

print("rollout steps:", len(history))


Пропуск rollout: runtime недоступен.
rollout steps: 0


In [9]:
if not history:
    print("Нет rollout-данных для графиков.")
else:
    steps = np.array([h["step"] for h in history], dtype=float)
    lin = np.array([h["linear"] for h in history], dtype=float)
    ang = np.array([h["angular"] for h in history], dtype=float)
    spd = np.array([h["speed"] for h in history], dtype=float)
    ctr = np.array([h["center"] for h in history], dtype=float)

    fig, axes = plt.subplots(2, 2, figsize=(12, 7), constrained_layout=True)
    axes[0, 0].plot(steps, lin)
    axes[0, 0].set_title("Linear cmd")
    axes[0, 0].set_xlabel("step")

    axes[0, 1].plot(steps, ang)
    axes[0, 1].set_title("Angular cmd")
    axes[0, 1].set_xlabel("step")

    axes[1, 0].plot(steps, spd)
    axes[1, 0].set_title("Speed m/s")
    axes[1, 0].set_xlabel("step")

    axes[1, 1].plot(steps, ctr)
    axes[1, 1].set_title("Line center sensor S3")
    axes[1, 1].set_xlabel("step")

    plt.show()


Нет rollout-данных для графиков.


## Что это демонстрирует

- управление машинкой идет через ROS2 topic `/cmd_vel`;
- сенсоры читаются из ROS2 топиков (`line_tracker`, `speedometer`);
- простая обученная модель формирует рабочий steering-профиль для трассы.
